Before EGU 2026, I decided to re-make the passage of OPT operator derivation since we need to take care of staggered grids etc. which are not easy to handle with the current version

# Big change!!

pointsInSpace, pointsInTime should be real (before they were -1 but now it should be 2,3,4 ...)

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")

include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [2]:
# Since this is the developing version, I do not use the module from flexOPT.jl

use some solid packages

In [3]:
include("../src/compactSymbolicFunctions/compactFunctionsArray.jl")
include("../src/compactSymbolicFunctions/BsplineHelpers.jl")
include("../src/CompactSymbolicFunctions/TaylorExpansionHelpers.jl")
include("../src/CompactSymbolicFunctions/integralWYYKK.jl")

WYYKKIntegralPureSymbolic (generic function with 1 method)

In [4]:
include("../src/motorsOPT/others.jl")
include("../src/motorsOPT/famousEquations.jl")

famousEquation (generic function with 15 methods)

input parameters

In [5]:
famousEquationType="2DacousticTime"
Δ = (1.0,1.0,1.0)

# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# new parameters for interpolated Taylor expansion
# μ points should be distributed from y_min+offsetμFromExtremitiesInΔy*Δy to y_max-offsetμFromExtremitiesInΔy*Δy
pointsμInSpace = 1
pointsμInTime = 1
offsetμFromExtremitiesInΔyInSpace=1
offsetμFromExtremitiesInΔyInTime=1

YorderBspace=-1
YorderBtime=-1
supplementaryOrder=2

2

We also propose different ways of boundary treatment (ghosting: Neumann/Dirichlet; same number of points etc.)

future functions

Starting the OPT derivation

In [13]:

Δnum = SVector(Δ)

concreteParametersForOPTConstruction = @strdict famousEquationType Δnum orderBtime orderBspace YorderBtime YorderBspace supplementaryOrder pointsInSpace pointsInTime 

# construction of NamedTuples
trialFunctionsCharacteristics=(orderBtime=orderBtime,orderBspace=orderBspace,pointsInSpace=pointsInSpace,pointsInTime=pointsInTime)
TaylorOptions=(YorderBtime=YorderBtime,YorderBspace=YorderBspace,supplementaryOrder=supplementaryOrder,pointsμInSpace=pointsμInSpace,pointsμInTime=pointsμInTime,offsetμInΔyInSpace=offsetμFromExtremitiesInΔyInSpace,offsetμInΔyInTime=offsetμFromExtremitiesInΔyInTime)


(YorderBtime = -1, YorderBspace = -1, supplementaryOrder = 2, pointsμInSpace = 1, pointsμInTime = 1, offsetμInΔyInSpace = 1, offsetμInΔyInTime = 1)

In [14]:
equationCharacteristics=famousEquations(famousEquationType)
numbersOfTheSystem=numbersOfTheExpression(equationCharacteristics,trialFunctionsCharacteristics,TaylorOptions)
dependencies,ordersForSplines,configsTaylor=investigateDependencies(equationCharacteristics,numbersOfTheSystem,trialFunctionsCharacteristics,TaylorOptions)
bigα,varM=bigαFinder(equationCharacteristics,numbersOfTheSystem,ordersForSplines)
@show bigα,varM
@show configsTaylor


(bigα, varM) = (Any[Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))];;], Any[v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉])
configsTaylor = (numberGeometries = 1, multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = Any[SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14], availableμPoints = Any[SVector{3, Float64}[[2.0, 2.0, 2.0];;;]], availableμaxes = Any[([2.0], [2.0], [2.0])])


(numberGeometries = 1, multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = Any[SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14], availableμPoints = Any[SVector{3, Float64}[[2.0, 2.0, 2.0];;;]], availableμaxes = Any[([2.0], [2.0], [2.0])])

In [54]:

@kernel function windowContraction!(
    output::AbstractArray{Float32,3},   # Array{Float32,3} on GPU: (P,P,nalpha)
    C::AbstractArray{Float32,3},        # Array{Float32,3} on GPU: (P,L,P)
    int::AbstractArray{Float32,5},    # Array{Float32,5} on GPU: (nDim, int1max,int1max,int2max,int2max)
    table::AbstractArray{Int32,3}, # Array{Int32,3} on GPU: (2+nDim*2,L*L, nalpha)
    tablePoints::AbstractArray{Int32,2}, # Array{Int32,2} on GPU: (nDim,P)
    #sizeOf1DIntegrals::AbstractArray{Int32,1}, # max size of 1D WYYKK integrals
    P::Int32, L::Int32, nDim::Int32, nalpha::Int32, int1max::Int32,int2max::Int32
)

    (xᶜ, x, α) = @index(Global, NTuple)   # ← 3-D indices
    
    @inbounds begin
    
        if xᶜ ≤ size(output,1) && x ≤ size(output,2) && α ≤ size(output,3)

            acc = zero(eltype(output))

            for idx in 1:size(table, 2)  # iterate over all entries in table for this alpha
                i = table[1, idx, α]
                j = table[2, idx, α]
                

                if i >0 && j >0
                        
                    for μᶜ in 1:P
                        for μ in 1:P
                            prod_int = 1.0f0
                            for iDim in 1:nDim
                                
                                k = table[2+iDim, idx, α]
                                l = table[2+iDim+nDim, idx, α]
                                if k >0 && l >0
                    
                                    mᶜ=tablePoints[iDim,μᶜ]
                                    m=tablePoints[iDim,μ]

                                    prod_int *= int[iDim, mᶜ, m, k, l]
                        
                                end
                            end
                            acc += C[xᶜ, i, μᶜ] * C[x, j, μ] * prod_int  # <- corrected x' in first C
                        end
                    end
                
                end
            end

            output[xᶜ, x, α] = acc
        end
    end
end


windowContraction! (generic function with 4 methods)

In [ ]:
function constructAmatrix_develop(iConfigGeometry,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor,Δnum)
    
    
    #(coordinates,multiOrdersIndices,pointsIndices,multiPointsIndices,middleLinearν,Δ,varM,bigα,orderBspline,YorderBspline,NtypeofExpr,NtypeofFields;threads = 256,backend=backend)

    # This function is for one iConfigGeometry

    #region preparation

    @unpack nCoordinates,NtypeofExpr,NtypeofExpr,NtypeofFields = numbersOfTheSystem
    @unpack multiOrdersIndices,availablePointsConfigurations,centrePointConfigurations,availableμPoints,availableμaxes = configsTaylor
    @unpack orderBspline, YorderBspline = ordersForSplines


    @show pointsIndices=availablePointsConfigurations[iConfigGeometry] # CartesianIndices
    @show middleLinearν=centrePointConfigurations[iConfigGeometry] # scalar
    @show μPoints = availableμPoints[iConfigGeometry] # Array(SVector)
    @show μaxes = availableμaxes[iConfigGeometry]
    @show size(μPoints)
    @show pointν = pointsIndices[middleLinearν] # SVector
    
    # this is fully GPU optimised version of ASymbolic 
    
    nPoints = length(pointsIndices)
    nLs = length(multiOrdersIndices)

    # 

    L_MINUS_N = multiOrdersIndices
    L_MINUS_N = L_MINUS_N .-L_MINUS_N[1] 
    # here L_MINUS_N is truly \mathbf{l}-\mahtbf{n} ∈ \mathbb{Z}_{≥0}

    #endregion

    #region we compute the integral of WYYKK in 1D domain 
    
    # look at the debug1DKernelIntegral.ipynb    
    
    coefWYYKK = [] # Array of (l_n_max+1,lᶜ_nᶜ_max+1,length(μs),length(μᶜs),length(ν) ) times nCoordinates
    for iCoord ∈ 1:nCoordinates
        maxNodes = pointsIndices[end][iCoord]
        nodesFromOne = [1,2,3] # ∈ Z like [1,2,3], an array of integers collect(1:1:Npoints) (nothing else!!)
        ν = (pointν[iCoord]) # this should be one point (for the moment)
        lᶜ_nᶜ_max = L_MINUS_N[end][iCoord] # variable
        l_n_max = L_MINUS_N[end][iCoord] # field
        params = (orderBspline1D=orderBspline[iCoord], YorderBspline1D=YorderBspline[iCoord], μᶜs=μaxes[iCoord], μs=μaxes[iCoord], maxNode = pointsIndices[end][iCoord], ν =(pointν[iCoord]), lᶜ_nᶜ_max=lᶜ_nᶜ_max, l_n_max=l_n_max,  Δ=Δnum[iCoord])
        tmpCoefWYYKK = WYYKKIntegralNumerical(params) 
        push!(coefWYYKK,tmpCoefWYYKK)
    end

    #endregion

    #region get CˡηGlobal (for ν)

    coefInversionDict = @strdict multiOrdersIndices pointsIndices μpointsIndices=μPoints Δ=Δnum

    @show typeof(μPoints), μPoints[1], typeof(pointsIndices)
    output=myProduceOrLoad(TaylorCoefInversion,coefInversionDict,"taylorCoefInv")
    @show Cˡη=output["CˡηGlobal"]

    #endregion 

    #region 
    # the order is: (νᶜ,) ν, i, j  here
    Ajiννᶜ = Array{Num,4}(undef,nPoints,nPoints,NtypeofFields,NtypeofExpr)

    # but this will already include bigα so the coefficients for each α_{nn'ji} should be given here
    #endregion


    #region useful LinearIndices conversion functions
    
    LI_points = LinearIndices(pointsIndices)
    LI_L_MINUS_N_plus_1 = LinearIndices(L_MINUS_N.+vec2car(ones(Int,nCoordinates)))
  
    #endregion


    #region make the table for each (x',x,n',n) (x=η+μ)
    @show nTotalSmallα = sum(length(bigα[iExpr, iField]) for iExpr in 1:NtypeofExpr, iField in 1:NtypeofFields)


    return coefWYYKK,Cˡη

end


constructAmatrix_develop (generic function with 1 method)

In [53]:
coef,Cˡη = constructAmatrix_develop(1,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor,Δnum)
    

pointsIndices = availablePointsConfigurations[iConfigGeometry] = SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]
middleLinearν = centrePointConfigurations[iConfigGeometry] = 14
μPoints = availableμPoints[iConfigGeometry] = SVector{3, Float64}[[2.0, 2.0, 2.0];;;]
μaxes = availableμaxes[iConfigGeometry] = ([2.0], [2.0], [2.0])
size(μPoints) = (1, 1, 1)
pointν = pointsIndices[middleLinearν] = [2, 2, 2]
(typeof(μPoints), μPoints[1], typeof(pointsIndices)) = (Array{SVector{3, Float64}, 3}, [2.0, 2.0, 2.0], Array{SVector{3, Int64}, 3})
Cˡη = output["CˡηGlobal"] = [6.661338147750939e-16 -1.1102230246251565e-15 -4.9960036108132044e-15 -7.328988471630057e-17 -4.138394237017872e-16 -7.771561172376096e-16 -1.2212453270876722e-15 -2.220446049250313e

(Any[[1.0 -6.661338147750939e-16 … -6.045317988566111e-76 0.002777777777777778; -6.661338147750939e-16 0.16666666666666297 … 0.011111111111111112 -1.2263359348234111e-75; … ; -6.045317988566111e-76 0.011111111111111112 … 0.000992063492063492 -1.8870028292881362e-75; 0.002777777777777778 -1.2263359348234111e-75 … -1.8870028292881362e-75 3.8580246913580246e-5;;;;;], [1.0 -6.661338147750939e-16 … -6.045317988566111e-76 0.002777777777777778; -6.661338147750939e-16 0.16666666666666297 … 0.011111111111111112 -1.2263359348234111e-75; … ; -6.045317988566111e-76 0.011111111111111112 … 0.000992063492063492 -1.8870028292881362e-75; 0.002777777777777778 -1.2263359348234111e-75 … -1.8870028292881362e-75 3.8580246913580246e-5;;;;;], [1.0 -6.661338147750939e-16 … -6.045317988566111e-76 0.002777777777777778; -6.661338147750939e-16 0.16666666666666297 … 0.011111111111111112 -1.2263359348234111e-75; … ; -6.045317988566111e-76 0.011111111111111112 … 0.000992063492063492 -1.8870028292881362e-75; 0.0027777

In [35]:
coef[1]

5×5×1×1×1 Array{Float64, 5}:
[:, :, 1, 1, 1] =
  1.0          -6.66134e-16   0.0833333    -6.04532e-76   0.00277778
 -6.66134e-16   0.166667     -2.2454e-76    0.0111111    -1.22634e-75
  0.0833333    -2.2454e-76    0.0166667    -2.45267e-75   0.000744048
 -6.04532e-76   0.0111111    -2.45267e-75   0.000992063  -1.887e-75
  0.00277778   -1.22634e-75   0.000744048  -1.887e-75     3.85802e-5

In [39]:
size(Cˡη)

(27, 125, 1)

In [51]:
reshape(Cˡη[:,2,1],(3,3,3))

3×3×3 Array{Float64, 3}:
[:, :, 1] =
 -1.11022e-15  0.0          -3.33067e-16
  1.11022e-15  6.21725e-15  -1.11022e-15
 -7.77156e-16  2.88658e-15   0.0

[:, :, 2] =
 -2.22045e-16  -0.486486  -3.33067e-15
  3.10862e-15   0.0        1.77636e-15
  8.88178e-16   0.486486   1.33227e-15

[:, :, 3] =
  3.33067e-16   1.55431e-15   2.22045e-16
 -8.88178e-16   0.0          -1.33227e-15
  4.44089e-16  -1.33227e-15   4.44089e-16

In [19]:
function AmatrixSemiSymbolicGPU(iConfigGeometry,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor)
    
    
    #(coordinates,multiOrdersIndices,pointsIndices,multiPointsIndices,middleLinearν,Δ,varM,bigα,orderBspline,YorderBspline,NtypeofExpr,NtypeofFields;threads = 256,backend=backend)

    # This function is for one iConfigGeometry

    #region preparation

    @unpack nCoordinates = numbersOfTheSystem

    @unpack multiOrdersIndices,availablePointsConfigurations,availableμPoints = configsTaylor

  

    @show pointsIndices=availablePointsConfigurations[iConfigGeometry] # CartesianIndices
    @show middleLinearν=centrePointConfigurations[iConfigGeometry] # scalar
    @show μPoints = availableμPoints[iConfigGeometry] # Array(SVector)
    @show pointν = pointsIndices[middleLinearν] # SVector
    

    # this is fully GPU optimised version of ASymbolic 
    
    nPoints = length(pointsIndices)
    nLs = length(multiOrdersIndices)

    # 

    L_MINUS_N = multiOrdersIndices
    L_MINUS_N = L_MINUS_N .-L_MINUS_N[1]
    # here L_MINUS_N is truly \mathbf{l}-\mahtbf{n} ∈ \mathbb{Z}_{≥0}

    #endregion

    #region we compute the integral of WYYKK in 1D domain (μ ∈ \mathbb{Z}/2 has not yet been implemented)

    



    integral1DWYYKK = Array{Any,1}(undef,nCoordinates)
    modifiedμ=Array{Any,1}(undef,nCoordinates)
    for iCoord in eachindex(coordinates) # for each 
        integralParams = @strdict oB =orderBspline[iCoord] oWB = YorderBspline[iCoord] νCoord=pointsIndices[middleLinearν][iCoord] LCoord = multiPointsIndices[end][iCoord] ΔCoord=Δ[iCoord] l_n_max=L_MINUS_N[end][iCoord]
        output = myProduceOrLoad(getIntegralWYYKK,integralParams,"intKernel")
        integral1DWYYKK[iCoord] = output["intKernelforνLΔ"]
        modifiedμ[iCoord] = output["modμ"] # this can be still 'nothing'
    end



    #endregion

    #region get CˡηGlobal (for ν)

    coefInversionDict = @strdict coordinates multiOrdersIndices pointsIndices Δ YorderBspline modifiedμ

    #@show coordinates multiOrdersIndices pointsIndices Δ YorderBspline modifiedμ

    output=myProduceOrLoad(TaylorCoefInversion,coefInversionDict,"taylorCoefInv")
    Cˡη=output["CˡηGlobal"]

    #endregion 

    #region objectives (Ajiννᶜ)

    # the order is: (νᶜ,) ν, i, j  here
    Ajiννᶜ = Array{Num,4}(undef,nPoints,nPoints,NtypeofFields,NtypeofExpr)

    # but this will already include bigα so the coefficients for each α_{nn'ji} should be given here

    #endregion

    #region useful LinearIndices conversion functions
    
    LI_points = LinearIndices(pointsIndices)
    LI_L_MINUS_N_plus_1 = LinearIndices(L_MINUS_N.+vec2car(ones(Int,nCoordinates)))
  
    #endregion

    #region make the table for each (x',x,n',n) (x=η+μ)
    nTotalSmallα = sum(length(bigα[iExpr, iField]) for iExpr in 1:NtypeofExpr, iField in 1:NtypeofFields)

    tableForLoop = Array{Int32,3}(undef,2+nCoordinates*2,nLs*nLs,nTotalSmallα) # for every α it will give the l and l-n
    
    fill!(tableForLoop, 0)
    indexLinearα = 1
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        
        α = bigα[iExpr,iField]
        for eachα ∈ α
            nᶜ = eachα.nᶜ - vec2car(ones(Int,nCoordinates))
            n = eachα.n - vec2car(ones(Int,nCoordinates))
            #nodeValue = eachα.node # not important at this point
            # Available indices
            Lᶜ_avail = (nᶜ .+ L_MINUS_N) ∩ L_MINUS_N
            L_avail = (n .+ L_MINUS_N) ∩ L_MINUS_N
            Lᶜ_Nᶜ_avail = Lᶜ_avail .- nᶜ
            L_N_avail = L_avail .- n
            
            iL=1
            for lᶜ ∈ Lᶜ_avail, l ∈ L_avail
                
                tableForLoop[1,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ+vec2car(ones(Int,nCoordinates))]
                tableForLoop[2,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[3,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[4,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l-n+vec2car(ones(Int,nCoordinates))]
                tmplᶜ_nᶜ = lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))
                tmpl_n = l-n+vec2car(ones(Int,nCoordinates))
                for iCoord ∈ 1:nCoordinates
                    tableForLoop[2+iCoord,iL,indexLinearα] = tmplᶜ_nᶜ[iCoord]
                    tableForLoop[2+iCoord+nCoordinates,iL,indexLinearα] = tmpl_n[iCoord]
                end

                iL += 1
            end
       
            indexLinearα += 1
        end
    end


    #endregion

    #region make a dictionary for μ ∈ pointsIndices and its linearised version

    tableForPoints = Array{Int32,2}(undef,nCoordinates,nPoints)

    for μ ∈ pointsIndices, iCoord ∈ 1:nCoordinates
        linearisedμ = LI_points[vec2car(μ)]
        tableForPoints[iCoord,linearisedμ]=vec2car(μ)[iCoord]
    end

    #endregion

    #region adapt the arrays to the GPU backend
    tableForPoints_gpu = Adapt.adapt(backend,tableForPoints)
    tableForLoop_gpu = Adapt.adapt(backend,tableForLoop)
    C_gpu = Adapt.adapt(backend,Float32.(Cˡη))


    # Collect the size of each array
    all_sizes = collect.(size.(integral1DWYYKK))  # vector of tuples

    # Element-wise maximum across all dimensions
    max_size = map((xs...) -> maximum(xs), all_sizes...)

    int_total_float32 = Array{Float32,5}(undef, nCoordinates, max_size...)
    int_total_float32 .= 0.f0
    for iCoord in 1:nCoordinates
        small_size = size(integral1DWYYKK[iCoord])
        tmpMatrix = Float32.(integral1DWYYKK[iCoord])
        tmpRange = CartesianIndices(tmpMatrix)
        int_total_float32[iCoord,tmpRange] = tmpMatrix
    end
    int_gpu = Adapt.adapt(backend,int_total_float32)
    


    output_gpu = Adapt.adapt(backend, zeros(Float32, nPoints, nPoints, nTotalSmallα))
    
    #


    #endregion
    
    #region launch GPU computation

    # scalars in Int32
    P      = Int32(nPoints)
    L      = Int32(nLs)
    nDim   = Int32(nCoordinates)
    nα     = Int32(nTotalSmallα)
    int1max = Int32(max_size[1])
    int2max = Int32(max_size[2])

    #@show typeof(output_gpu)         # must be MtlArray{Float32,3}
    #@show typeof(C_gpu)              # must be MtlArray{Float32,3}
    #@show typeof(int_gpu)            # must be MtlArray{Float32,5}
    #@show typeof(tableForLoop_gpu)   # must be MtlArray{Int32,3}
    #@show typeof(tableForPoints_gpu) # must be MtlMatrix{Int32}
    #@show typeof(P) typeof(L) typeof(nDim) typeof(nα) typeof(int1max) typeof(int2max)

    kernel! = windowContraction!(backend,(8,8,8))#,128,size(output_gpu))
    kernel!(output_gpu, C_gpu, int_gpu, tableForLoop_gpu,tableForPoints_gpu,
       P,L,nDim,nα,int1max,int2max,ndrange=size(output_gpu))
    KernelAbstractions.synchronize(backend)

    # Here output_gpu[x',x,eachα] ∑μ' ∑μ ∑l' ∑ l C[x',l',μ'] C[x,l,μ] ∏_iCoord K[iCoord][μ',μ,l'-n',l-n]
    # with (n',n) depends on eachα and x=η+μ

    @show "GPU computation of Ajiννᶜ: done"
    output = Adapt.adapt(CPU(), output_gpu)
    #endregion


    #region contruct Ulocal

    # the order is: (νᶜ,) ν, i, j  here

    Ulocal = Array{Num,2}(undef,length(pointsIndices),NtypeofFields)
    for iField in eachindex(fields)
        newstring=split(string(fields[iField]),"(")[1]
        Ulocal[:,iField]=Symbolics.variables(Symbol(newstring),1:length(pointsIndices))
    end

    #endregion

    #region make Ajiννᶜ and AjiννᶜU symbolically (which we will soon remove!)
    Ajiννᶜ = Array{Num,3}(undef,length(pointsIndices),NtypeofFields,NtypeofExpr)
    AjiννᶜU = Array{Num,1}(undef,NtypeofExpr)
    
    # this is the cost function for ν point so the number of elements is just the number of expressions (governing equations)
    Ajiννᶜ .= 0
    AjiννᶜU .= 0
    indexLinearα = 1
   
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        α = bigα[iExpr,iField]
        for eachα ∈ α
            @show nodeValue=eachα.node
            for x ∈ pointsIndices
                xLinear = LI_points[vec2car(x)]
                
                localmapηᶜ=Dict()
                for iVar ∈ eachindex(vars)
                    localmapηᶜ[vars[iVar]]=varM[iVar,xLinear][]
                end
                for xᶜ ∈ pointsIndices
                    xᶜLinear = LI_points[vec2car(xᶜ)]
                    U_HERE = Ulocal[xᶜLinear,iField]                    
                    substitutedValue = substitute(nodeValue, localmapηᶜ)
                    Ajiννᶜ[xᶜLinear,iField,iExpr] += Float64(output[xᶜLinear,xLinear,indexLinearα])*substitutedValue
                    AjiννᶜU[iExpr]+= Ajiννᶜ[xᶜLinear,iField,iExpr] * U_HERE
                end

            end
            indexLinearα += 1
        end

    end


    #endregion


    return AjiννᶜU,Ulocal, output
end 

AmatrixSemiSymbolicGPU (generic function with 1 method)

In [23]:
bigα

1×1 Matrix{Any}:
 Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))]

In [24]:
configsTaylor.availablePointsConfigurations[1]

3×3×3 Array{SVector{3, Int64}, 3}:
[:, :, 1] =
 [1, 1, 1]  [1, 2, 1]  [1, 3, 1]
 [2, 1, 1]  [2, 2, 1]  [2, 3, 1]
 [3, 1, 1]  [3, 2, 1]  [3, 3, 1]

[:, :, 2] =
 [1, 1, 2]  [1, 2, 2]  [1, 3, 2]
 [2, 1, 2]  [2, 2, 2]  [2, 3, 2]
 [3, 1, 2]  [3, 2, 2]  [3, 3, 2]

[:, :, 3] =
 [1, 1, 3]  [1, 2, 3]  [1, 3, 3]
 [2, 1, 3]  [2, 2, 3]  [2, 3, 3]
 [3, 1, 3]  [3, 2, 3]  [3, 3, 3]

In [25]:
configsTaylor.availableμPoints[1]

1×1×1 Array{SVector{3, Float64}, 3}:
[:, :, 1] =
 [2.0, 2.0, 2.0]

In [26]:
typeof(configsTaylor.availablePointsConfigurations[1])

Array{SVector{3, Int64}, 3}

In [27]:
configsTaylor.availablePointsConfigurations[1][configsTaylor.centrePointConfigurations[1]]

3-element SVector{3, Int64} with indices SOneTo(3):
 2
 2
 2

In [28]:
@show numbersOfTheSystem.pointsμUsed
@show numbersOfTheSystem.offsetsμUsed

numbersOfTheSystem.pointsμUsed = [1, 1, 1]
numbersOfTheSystem.offsetsμUsed = [1.0, 1.0, 1.0]


3-element Vector{Float64}:
 1.0
 1.0
 1.0